# 04 — Evaluate the Mizo NER model (v2) on the test set

Full evaluation of `models/mizo_ner_v2` on the 44,118-sentence held-out split.
Produces every number Section 6 of the paper needs, plus regenerated figures.

Test labels are silver-standard, produced by the same projection pipeline as
training. These scores measure agreement with the projection, not accuracy
against human annotation.

**Run from the repository root.** Kernel: `Python (tka)`. About 10 minutes.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, sys, time
from collections import Counter, defaultdict
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

DATA  = ROOT / "data" / "processed" / "bio_v2"
MODEL = ROOT / "models" / "mizo_ner_v2"
RES   = ROOT / "results" / "ner"
FIGS  = ROOT / "paper" / "figures"
RES.mkdir(parents=True, exist_ok=True); FIGS.mkdir(parents=True, exist_ok=True)

for p in (DATA / "mizo_ner_test.json", MODEL / "config.json",
          RES / "training_v2.json"):
    print(("  ok   " if p.exists() else "  MISS ") + str(p.relative_to(ROOT)))
    if not p.exists():
        sys.exit("Run 03_ner_training_v2.ipynb first")

# take MAX_LEN from the training record so evaluation cannot drift from training
train_cfg = json.load(open(RES / "training_v2.json"))
MAX_LEN = train_cfg["max_len"]
BATCH   = 128
print(f"\nMAX_LEN from training record: {MAX_LEN}")
print(f"Training took {train_cfg['training_hours']} h, best dev F1 "
      f"{max(h['f1'] for h in train_cfg['history']):.4f}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Repo root: C:\Users\Haulai\mizo-ner
  ok   data\processed\bio_v2\mizo_ner_test.json
  ok   models\mizo_ner_v2\config.json
  ok   results\ner\training_v2.json

MAX_LEN from training record: 96
Training took 2.57 h, best dev F1 0.8727
Device: cuda


## Cell 2: Load model and test data

In [2]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(str(MODEL))
model = AutoModelForTokenClassification.from_pretrained(str(MODEL)).to(device).eval()
id2tag = {int(k): v for k, v in model.config.id2label.items()}
print(f"Labels: {len(id2tag)}")

test = json.load(open(DATA / "mizo_ner_test.json", encoding="utf-8"))
print(f"Test sentences: {len(test):,}")
gold_entities = sum(1 for r in test for t in r["tags"] if t.startswith("B-"))
print(f"Gold entities : {gold_entities:,}")

Labels: 23
Test sentences: 44,118
Gold entities : 58,598


## Cell 3: Predict

Predictions are read back at word level: the label of a word is the label
predicted for its first subword, matching how training assigned labels.

In [3]:
def predict(records, batch_size=BATCH):
    all_true, all_pred = [], []
    t0 = time.time()
    for i in range(0, len(records), batch_size):
        chunk = records[i:i + batch_size]
        toks = [r["tokens"] for r in chunk]
        enc = tokenizer(toks, is_split_into_words=True, max_length=MAX_LEN,
                        padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        preds = torch.argmax(logits, dim=2).cpu().numpy()
        for b, rec in enumerate(chunk):
            wid, seen = enc.word_ids(batch_index=b), set()
            n = len(rec["tokens"])
            row = ["O"] * n
            for pos, w in enumerate(wid):
                if w is not None and w not in seen:
                    seen.add(w)
                    row[w] = id2tag[int(preds[b][pos])]
            all_true.append(list(rec["tags"]))
            all_pred.append(row)
        if (i // batch_size) % 50 == 0:
            print(f"  {i:,} / {len(records):,}")
    print(f"Done in {(time.time()-t0)/60:.1f} min")
    return all_true, all_pred

y_true, y_pred = predict(test)
assert all(len(a) == len(b) for a, b in zip(y_true, y_pred))
print(f"\nSequences: {len(y_true):,}   tokens: {sum(len(s) for s in y_true):,}")

  0 / 44,118
  6,400 / 44,118
  12,800 / 44,118
  19,200 / 44,118
  25,600 / 44,118
  32,000 / 44,118
  38,400 / 44,118
Done in 1.2 min

Sequences: 44,118   tokens: 497,148


## Cell 4: Overall scores

In [4]:
from seqeval.metrics import (classification_report, f1_score,
                             precision_score, recall_score)

flat_t = [t for s in y_true for t in s]
flat_p = [t for s in y_pred for t in s]

tok_acc_all = np.mean([a == b for a, b in zip(flat_t, flat_p)])
ent_idx = [i for i, t in enumerate(flat_t) if t != "O"]
tok_acc_ent = np.mean([flat_t[i] == flat_p[i] for i in ent_idx])

overall = {
    "token_accuracy_all":    round(float(tok_acc_all) * 100, 2),
    "token_accuracy_entity": round(float(tok_acc_ent) * 100, 2),
    "precision_micro": round(float(precision_score(y_true, y_pred)), 4),
    "recall_micro":    round(float(recall_score(y_true, y_pred)), 4),
    "f1_micro":        round(float(f1_score(y_true, y_pred)), 4),
    "precision_macro": round(float(precision_score(y_true, y_pred, average="macro")), 4),
    "recall_macro":    round(float(recall_score(y_true, y_pred, average="macro")), 4),
    "f1_macro":        round(float(f1_score(y_true, y_pred, average="macro")), 4),
}

OLD = {"token_accuracy_all": 97.84, "token_accuracy_entity": 82.07,
       "precision_micro": 0.8378, "recall_micro": 0.8686, "f1_micro": 0.8529,
       "precision_macro": 0.6703, "recall_macro": 0.6993, "f1_macro": 0.6803}

print(f"{'Metric':<26}{'old':>10}{'new':>10}{'change':>10}")
print("-" * 56)
for k, v in overall.items():
    o = OLD[k]
    print(f"{k:<26}{o:>10.4f}{v:>10.4f}{v-o:>+10.4f}")

Metric                           old       new    change
--------------------------------------------------------
token_accuracy_all           97.8400   97.6500   -0.1900
token_accuracy_entity        82.0700   87.7500   +5.6800
precision_micro               0.8378    0.8653   +0.0275
recall_micro                  0.8686    0.8827   +0.0141
f1_micro                      0.8529    0.8739   +0.0210
precision_macro               0.6703    0.7155   +0.0452
recall_macro                  0.6993    0.7206   +0.0213
f1_macro                      0.6803    0.7141   +0.0338


## Cell 5: Per-type breakdown

In [5]:
rep = classification_report(y_true, y_pred, output_dict=True, digits=4)
per_type = {k: v for k, v in rep.items()
            if k not in ("micro avg", "macro avg", "weighted avg")}

OLD_F1 = {"PERSON":0.9045,"GPE":0.8456,"WORK_OF_ART":0.8330,"ORG":0.7760,
          "NORP":0.7603,"LANGUAGE":0.7432,"LAW":0.6466,"LOC":0.6443,
          "FAC":0.5564,"EVENT":0.4222,"PRODUCT":0.3514}

rows = sorted(per_type.items(), key=lambda kv: -kv[1]["f1-score"])
print(f"{'Entity':<14}{'Prec':>9}{'Recall':>9}{'F1':>9}{'Support':>10}{'old F1':>9}{'change':>9}")
print("-" * 69)
for name, m in rows:
    o = OLD_F1.get(name)
    delta = f"{m['f1-score']-o:+.4f}" if o else "   n/a"
    ostr  = f"{o:.4f}" if o else "  n/a"
    print(f"{name:<14}{m['precision']:>9.4f}{m['recall']:>9.4f}"
          f"{m['f1-score']:>9.4f}{int(m['support']):>10,}{ostr:>9}{delta:>9}")

Entity             Prec   Recall       F1   Support   old F1   change
---------------------------------------------------------------------
PERSON           0.9066   0.9275   0.9169    32,629   0.9045  +0.0124
GPE              0.8663   0.8921   0.8790    11,495   0.8456  +0.0334
WORK_OF_ART      0.8342   0.8058   0.8198       381   0.8330  -0.0132
ORG              0.7817   0.8020   0.7917    10,196   0.7760  +0.0157
NORP             0.8147   0.7379   0.7744     1,942   0.7603  +0.0141
LANGUAGE         0.6858   0.8747   0.7688       479   0.7432  +0.0256
LOC              0.6906   0.7014   0.6960       700   0.6443  +0.0517
LAW              0.6282   0.7206   0.6712        68   0.6466  +0.0246
FAC              0.6487   0.5535   0.5974       327   0.5564  +0.0410
EVENT            0.5000   0.5778   0.5361        90   0.4222  +0.1139
PRODUCT          0.5132   0.3333   0.4042       291   0.3514  +0.0528


## Cell 6: Frequency and performance

The paper argues that per-type performance is governed by annotation frequency
rather than linguistic difficulty. Recompute the correlation on the new results.

In [6]:
from scipy.stats import pearsonr, spearmanr

names = [n for n, _ in rows]
f1s   = np.array([per_type[n]["f1-score"] for n in names])
sups  = np.array([per_type[n]["support"] for n in names], dtype=float)

rho, p_rho = spearmanr(sups, f1s)
r_log, p_log = pearsonr(np.log10(sups), f1s)
r_raw, p_raw = pearsonr(sups, f1s)

print(f"Spearman rho          : {rho:.4f}  (p = {p_rho:.4f})")
print(f"Pearson r, log support: {r_log:.4f}  (p = {p_log:.4f})")
print(f"Pearson r, raw support: {r_raw:.4f}  (p = {p_raw:.4f})")
print(f"\nPaper currently states: Spearman 0.75 (p=0.008), Pearson log 0.70 (p=0.016)")

Spearman rho          : 0.8364  (p = 0.0013)
Pearson r, log support: 0.7214  (p = 0.0122)
Pearson r, raw support: 0.6095  (p = 0.0465)

Paper currently states: Spearman 0.75 (p=0.008), Pearson log 0.70 (p=0.016)


## Cell 7: Type confusions

In [7]:
conf = Counter()
for ts, ps in zip(y_true, y_pred):
    for t, p in zip(ts, ps):
        gt = t[2:] if t != "O" else "O"
        gp = p[2:] if p != "O" else "O"
        if gt != gp:
            conf[(gt, gp)] += 1

ent_conf = {k: v for k, v in conf.items() if k[0] != "O" and k[1] != "O"}
top = sorted(ent_conf.items(), key=lambda kv: -kv[1])[:10]

print("Most frequent entity-type confusions (token level)")
print(f"{'gold -> predicted':<30}{'count':>8}")
print("-" * 38)
for (g, p), n in top:
    print(f"{g + '  ->  ' + p:<30}{n:>8,}")

missed = sum(v for k, v in conf.items() if k[1] == "O")
spurious = sum(v for k, v in conf.items() if k[0] == "O")
print(f"\nentity token predicted as O : {missed:>8,}")
print(f"O predicted as an entity    : {spurious:>8,}")

Most frequent entity-type confusions (token level)
gold -> predicted                count
--------------------------------------
ORG  ->  PERSON                  1,058
PERSON  ->  ORG                    872
GPE  ->  PERSON                    581
PERSON  ->  GPE                    547
GPE  ->  ORG                       391
ORG  ->  GPE                       379
NORP  ->  LANGUAGE                 187
NORP  ->  PERSON                   153
ORG  ->  NORP                       86
PRODUCT  ->  ORG                    81

entity token predicted as O :    2,355
O predicted as an entity    :    3,451


## Cell 8: Save results

In [8]:
out = {
    "model": str(MODEL.relative_to(ROOT)),
    "max_len": MAX_LEN,
    "test_sentences": len(test),
    "gold_entities": gold_entities,
    "overall": overall,
    "overall_previous_run": OLD,
    "per_type": {n: {"precision": round(m["precision"], 4),
                     "recall":    round(m["recall"], 4),
                     "f1":        round(m["f1-score"], 4),
                     "support":   int(m["support"]),
                     "f1_previous": OLD_F1.get(n)} for n, m in per_type.items()},
    "correlation": {"spearman_rho": round(float(rho), 4),
                    "spearman_p": round(float(p_rho), 4),
                    "pearson_log_r": round(float(r_log), 4),
                    "pearson_log_p": round(float(p_log), 4)},
    "confusions_top10": [{"gold": g, "pred": p, "count": int(n)} for (g, p), n in top],
    "entity_tokens_missed": int(missed),
    "spurious_entity_tokens": int(spurious),
}
json.dump(out, open(RES / "evaluation_v2.json", "w"), indent=2)

with open(RES / "test_report_v2.txt", "w", encoding="utf-8") as f:
    f.write(classification_report(y_true, y_pred, digits=4))
print(f"-> {(RES/'evaluation_v2.json').relative_to(ROOT)}")
print(f"-> {(RES/'test_report_v2.txt').relative_to(ROOT)}")

-> results\ner\evaluation_v2.json
-> results\ner\test_report_v2.txt


## Cell 9: Regenerate the paper figures

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    "font.family": "serif", "font.size": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "figure.dpi": 300,
})
GREY = "0.45"

# --- F1 against support ---
fig, ax = plt.subplots(figsize=(5.0, 3.0))
ax.scatter(sups, f1s, s=34, facecolors="none", edgecolors="black", lw=1.0, zorder=3)
ls = np.log10(sups); m_, b_ = np.polyfit(ls, f1s, 1)
xs = np.linspace(ls.min() - 0.2, ls.max() + 0.2, 100)
ax.plot(10**xs, m_ * xs + b_, "--", color=GREY, lw=1.0, zorder=2)
for n, x, y in zip(names, sups, f1s):
    ax.annotate(n, (x, y), textcoords="offset points", xytext=(7, 4), fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("Entity support in test set (log scale)")
ax.set_ylabel(r"Entity-level $F_1$")
ax.grid(True, alpha=0.22, lw=0.5); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.savefig(FIGS / "f1_vs_support.pdf"); plt.close(fig)

# --- confusions ---
fig, ax = plt.subplots(figsize=(4.6, 2.2))
lbl = [f"{g} $\\rightarrow$ {p}" for (g, p), _ in top[:5]]
cnt = [n for _, n in top[:5]]
y = np.arange(len(lbl))[::-1]
ax.barh(y, cnt, height=0.6, color="0.78", edgecolor="black", lw=0.7)
for yi, c in zip(y, cnt):
    ax.text(c + max(cnt) * 0.02, yi, f"{c:,}", va="center", fontsize=7.5)
ax.set_yticks(y); ax.set_yticklabels(lbl)
ax.set_xlabel("Misclassified tokens"); ax.set_xlim(0, max(cnt) * 1.18)
ax.grid(True, axis="x", alpha=0.22, lw=0.5); ax.set_axisbelow(True)
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.savefig(FIGS / "confusions.pdf"); plt.close(fig)

# --- training dynamics ---
hist = train_cfg["history"]
ep = [h["epoch"] for h in hist]
fig, axs = plt.subplots(1, 2, figsize=(6.4, 2.5))
axs[0].plot(ep, [h["train_loss"] for h in hist], "o-", color="black", ms=4, lw=1.2, label="Training")
axs[0].plot(ep, [h["dev_loss"] for h in hist], "s--", color=GREY, ms=4, lw=1.2, label="Development")
axs[0].set_xlabel("Epoch"); axs[0].set_ylabel("Cross-entropy loss")
axs[0].set_xticks(ep); axs[0].legend(frameon=False)
axs[1].plot(ep, [h["precision"] for h in hist], "s-", color=GREY, ms=4, lw=1.2, label="Precision")
axs[1].plot(ep, [h["recall"] for h in hist], "^--", color="0.15", ms=4, lw=1.2, label="Recall")
axs[1].plot(ep, [h["f1"] for h in hist], "o-", color="black", ms=4, lw=1.6, label=r"$F_1$")
axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("Score")
axs[1].set_xticks(ep); axs[1].legend(frameon=False, loc="lower right")
for a in axs:
    a.grid(True, alpha=0.22, lw=0.5); a.set_axisbelow(True)
    for s in ("top", "right"): a.spines[s].set_visible(False)
fig.savefig(FIGS / "training_dynamics.pdf"); plt.close(fig)

print("Figures rewritten in paper/figures/:")
for f in ("f1_vs_support.pdf", "confusions.pdf", "training_dynamics.pdf"):
    print(f"  {f}  ({(FIGS/f).stat().st_size/1024:.0f} KB)")

Figures rewritten in paper/figures/:
  f1_vs_support.pdf  (23 KB)
  confusions.pdf  (16 KB)
  training_dynamics.pdf  (21 KB)


## Cell 10: LaTeX tables for the paper

In [10]:
print("% ---- Table: overall ----")
print(f"Token accuracy (all tokens)     & {overall['token_accuracy_all']:.2f}\\% \\\\")
print(f"Token accuracy (entity tokens)  & {overall['token_accuracy_entity']:.2f}\\% \\\\")
for k, lab in [("precision_micro","Precision (micro)"), ("recall_micro","Recall (micro)"),
               ("f1_micro","$F_1$ (micro)"), ("precision_macro","Precision (macro)"),
               ("recall_macro","Recall (macro)"), ("f1_macro","$F_1$ (macro)")]:
    print(f"{lab:<32}& {overall[k]:.4f} \\\\")

print("\n% ---- Table: per type ----")
for n, m in rows:
    esc = n.replace("_", "\\_")
    print(f"{esc:<14}& {m['precision']:.4f} & {m['recall']:.4f} & "
          f"{m['f1-score']:.4f} & {int(m['support']):,} \\\\")

% ---- Table: overall ----
Token accuracy (all tokens)     & 97.65\% \\
Token accuracy (entity tokens)  & 87.75\% \\
Precision (micro)               & 0.8653 \\
Recall (micro)                  & 0.8827 \\
$F_1$ (micro)                   & 0.8739 \\
Precision (macro)               & 0.7155 \\
Recall (macro)                  & 0.7206 \\
$F_1$ (macro)                   & 0.7141 \\

% ---- Table: per type ----
PERSON        & 0.9066 & 0.9275 & 0.9169 & 32,629 \\
GPE           & 0.8663 & 0.8921 & 0.8790 & 11,495 \\
WORK\_OF\_ART & 0.8342 & 0.8058 & 0.8198 & 381 \\
ORG           & 0.7817 & 0.8020 & 0.7917 & 10,196 \\
NORP          & 0.8147 & 0.7379 & 0.7744 & 1,942 \\
LANGUAGE      & 0.6858 & 0.8747 & 0.7688 & 479 \\
LOC           & 0.6906 & 0.7014 & 0.6960 & 700 \\
LAW           & 0.6282 & 0.7206 & 0.6712 & 68 \\
FAC           & 0.6487 & 0.5535 & 0.5974 & 327 \\
EVENT         & 0.5000 & 0.5778 & 0.5361 & 90 \\
PRODUCT       & 0.5132 & 0.3333 & 0.4042 & 291 \\
